In [3]:

!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install --upgrade --force-reinstall llama-cpp-python --no-cache-dir
!pip install  bert-score datasets
!pip install llama-cpp-python



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 MB 261.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 144.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 262.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 187.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 189.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-py

In [4]:
import time
import pandas as pd
from llama_cpp import Llama
from datasets import load_dataset
from bert_score import score as bert_score
import re


In [5]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="jdavit/colombian-conflict-chat-Llama3.1",
	filename="unsloth.Q5_K_M.gguf",
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


unsloth.Q5_K_M.gguf:   0%|          | 0.00/5.73G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from /root/.cache/huggingface/hub/models--jdavit--colombian-conflict-chat-Llama3.1/snapshots/5e299041603a8a4bf63e0e0a3d416d0b6fdb0a48/./unsloth.Q5_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Colombian Conflict Chat Llama3.1
llama_model_loader: - kv   3:                           general.finetune str              = chat-Llama3.1
llama_model_loader: - kv   4:                           general.basename str              = colombian-conflict
llama_model_loader: - kv   5:                         general.size_label str              = 8.0B
llama_m

In [6]:
# Cargar dataset de HuggingFace
dataset = load_dataset("jdavit/colombian-conflict-SQA", split="test")


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2895 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30 [00:00<?, ? examples/s]

In [7]:
# Generar respuestas con el modelo Llama y medir latencia
records = []

for item in dataset:
    question = item["question"]
    context = item["context"]
    answer = item["answer"]

    prompt = f"""A continuación, se presenta una pregunta sobre el conflicto armado colombiano, junto con un contexto que proporciona información relevante. Escribe una respuesta que complete adecuadamente la solicitud.

Pregunta:
{question}

Contexto:
{context}

Respuesta:
"""

    # Medir tiempo de ejecución
    start_time = time.time()
    output = llm(
        prompt,
        max_tokens=150,
        echo=True
    )
    response = output["choices"][0]["text"].strip()
    latency = time.time() - start_time

    records.append({
        "question": question,
        "context": context,
        "expected_answer": answer,
        "predicted_answer": response,
        "latency": round(latency, 3)
    })

df = pd.DataFrame(records)


llama_perf_context_print:        load time =   14697.77 ms
llama_perf_context_print: prompt eval time =   14697.28 ms /   163 tokens (   90.17 ms per token,    11.09 tokens per second)
llama_perf_context_print:        eval time =   14349.68 ms /    91 runs   (  157.69 ms per token,     6.34 tokens per second)
llama_perf_context_print:       total time =   29151.67 ms /   254 tokens
Llama.generate: 48 prefix-match hit, remaining 150 prompt tokens to eval
llama_perf_context_print:        load time =   14697.77 ms
llama_perf_context_print: prompt eval time =   13694.98 ms /   150 tokens (   91.30 ms per token,    10.95 tokens per second)
llama_perf_context_print:        eval time =   10163.62 ms /    66 runs   (  153.99 ms per token,     6.49 tokens per second)
llama_perf_context_print:       total time =   23933.76 ms /   216 tokens
Llama.generate: 45 prefix-match hit, remaining 15 prompt tokens to eval
llama_perf_context_print:        load time =   14697.77 ms
llama_perf_context_print: 

In [8]:
# Normalización básica para F1
def normalize_text(text):
    return re.sub(r'\W+', ' ', text.strip().lower())

def compute_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * (precision * recall) / (precision + recall)

# Calcular métricas por fila
bert_precisions, bert_recalls, bert_f1s = bert_score(
    df["predicted_answer"].tolist(),
    df["expected_answer"].tolist(),
    lang="es",  # Asumiendo que las respuestas son en español
    verbose=True
)

df["bert_score"] = [round(f.item(), 3) for f in bert_f1s]
df["f1_score"] = [round(compute_f1(pred, gold), 3)
                  for pred, gold in zip(df["predicted_answer"], df["expected_answer"])]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 15.65 seconds, 1.92 sentences/sec


In [12]:
# Función para extraer la respuesta después de "Respuesta:"
def extract_answer(response):
    # Buscamos todo lo que está después de "Respuesta:"
    match = re.search(r"Respuesta:\s*(.*)", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        return response  # Si no encuentra "Respuesta:", devuelve el texto original

# Recorremos el DataFrame y corregimos las respuestas generadas
df["predicted_answer"] = df["predicted_answer"].apply(extract_answer)

# Verificamos los primeros registros para asegurarnos de que se realizó la corrección
df.head()

df.to_excel("Resultados_modelo_QA_llama.xlsx", index=False)
print("✅ Archivo guardado como evaluacion_modelo_QA_llama.xlsx")


✅ Archivo guardado como evaluacion_modelo_QA_llama.xlsx


In [13]:
df.head()


,question,context,expected_answer,predicted_answer,latency,bert_score,f1_score
0,¿Qué informe documenta la Operación Orión y su...,El informe 058‑CI‑01347 «Comuna 13: memorias d...,El informe 058‑CI‑01347 «Comuna 13: memorias d...,El informe describe la Operación Orión como un...,29.156,0.756,0.264
1,¿Qué informe analiza la relación entre los mon...,El informe 066‑CI‑00538 «Misión internacional ...,El informe «Misión internacional para la verif...,El informe identifica la relación entre los mo...,23.937,0.718,0.184
2,¿Cuál es la capital de Japón?,,"Lo siento, solo puedo responder preguntas rela...",Tokio es la capital de Japón.,2.982,0.688,0.185
3,¿Cuáles son las principales causas estructural...,La Comisión señala la concentración de la tier...,Las causas estructurales incluyen la marcada c...,Las principales causas estructurales identific...,18.331,0.739,0.221
4,¿Qué dice este informe 058-CI-00661 «Informe s...,El informe identifica como causas estructurale...,El informe atribuye el conflicto a causas estr...,El informe señala que las causas estructurales...,39.083,0.719,0.199
